# EDA-1 — Bằng chứng cho tiền xử lý ảnh

Notebook này giữ lại các kết quả cần để bàn giao sang tạo giao dịch: quy mô dữ liệu, lựa chọn patch, lọc nền và descriptor. Nó đọc các bảng EDA-1 đã lưu, không quét lại 70.000 ảnh và không sửa artifact TC.

**Không thuộc phạm vi:** chọn K/seed/minsup, chứng minh descriptor tốt nhất, hay đánh giá mô hình phân loại. So sánh patch 4×4 với 7×7 được ghi trong [`docs/UD2/TC_plan.md`](../../../docs/UD2/TC_plan.md); bảng kết quả EDA-1 hiện tại không chứa các dòng tính lại phép so sánh đó.


## 1. Môi trường và nguồn kết quả

In [1]:
using Pkg
using TOML
using Markdown

function find_repo_root(start=pwd())
    current = abspath(start)
    while true
        isfile(joinpath(current, "Project.toml")) && isfile(joinpath(current, "src", "EclatFIM.jl")) && return current
        parent = dirname(current)
        parent == current && error("Không tìm thấy thư mục gốc dự án từ $start")
        current = parent
    end
end

const REPO_ROOT = find_repo_root()
Pkg.activate(REPO_ROOT)
using EclatFIM
const CSV_READER = EclatFIM.FashionMNIST.FashionMNISTEDA2
const EDA1_DIR = joinpath(REPO_ROOT, "results", "fashion", "eda1")

function read_eda1(name)
    path = joinpath(EDA1_DIR, name)
    isfile(path) || error("Thiếu kết quả EDA-1: $path")
    return last(CSV_READER._read_csv(path))
end

function show_table(headers, rows)
    escape_cell(value) = replace(replace(string(value), "|" => "\\|"), "\n" => " ")
    lines = ["| " * join(escape_cell.(headers), " | ") * " |",
             "| " * join(fill("---", length(headers)), " | ") * " |"]
    append!(lines, ["| " * join(escape_cell.(row), " | ") * " |" for row in rows])
    display(Markdown.parse(join(lines, "\n")))
end

println("Project: ", REPO_ROOT)
println("EDA-1 outputs: ", EDA1_DIR)


  Activating 

Project: /run/media/fernir/Data/Documents/HCMUS/SEMESTER7/DataMining/TeamProject-FrequentItemsetMining

project at `/run/media/fernir/Data/Documents/HCMUS/SEMESTER7/DataMining/TeamProject-FrequentItemsetMining`



EDA-1 outputs: /run/media/fernir/Data/Documents/HCMUS/SEMESTER7/DataMining/TeamProject-FrequentItemsetMining/results/fashion/eda1


## 2. Kiểm tra dữ liệu cần cho các quyết định

In [2]:
metadata = TOML.parsefile(joinpath(EDA1_DIR, "10_run_metadata.toml"))
class_rows = read_eda1("01_class_counts.csv")
overview_rows = read_eda1("01_dataset_overview.csv")
overview = Dict(row["metric"] => row["value"] for row in overview_rows)

@assert parse(Int, overview["images"]) == 70_000
@assert parse(Int, overview["classes"]) == 10
@assert metadata["patch_size"] == 4
@assert metadata["q10_positive"] == 29
@assert length(class_rows) == 10
@assert all(parse(Int, row["images"]) == 7_000 for row in class_rows)

show_table(["Kiểm tra", "Kết quả"], [
    ["Số ảnh / số lớp", "$(overview["images"]) ảnh / $(overview["classes"]) lớp"],
    ["Ảnh mỗi lớp", "7.000"],
    ["Patch trong cấu hình EDA", "$(metadata["patch_rows"])×$(metadata["patch_columns"]), kích thước $(metadata["patch_size"])×$(metadata["patch_size"]), không chồng lấp"],
    ["Ngưỡng Q10+", string(metadata["q10_positive"])],
])


|                 Kiểm tra |                              Kết quả |
| ------------------------:| ------------------------------------:|
|          Số ảnh / số lớp |                   70000 ảnh / 10 lớp |
|              Ảnh mỗi lớp |                                7.000 |
| Patch trong cấu hình EDA | 7×7, kích thước 4×4, không chồng lấp |
|              Ngưỡng Q10+ |                                   29 |


## 3. Lựa chọn patch

Quyết định đưa sang TC là patch **4×4, stride 4**. Hồ sơ so sánh ghi nhận 30,73% patch 4×4 đồng nhất tối, so với 17,72% ở patch 7×7. Khi mức giảm patch xấp xỉ nhau (39,3%), tổn thất gradient được báo cáo là 5,92% với 4×4 và 14,00% với 7×7. Đây là bằng chứng cho việc giữ vùng cục bộ nhỏ khi lọc nền; nó **không** chứng minh codebook hoặc kết quả khai thác của patch 4×4 tốt hơn.

Các số trên được ghi trong `docs/UD2/TC_plan.md`; bảng dưới kiểm tra quy ước lưới 4×4 đang dùng ở bước sau.


In [3]:
patch_rows = read_eda1("04_C1_patch_grid.csv")
@assert length(patch_rows) == 49
@assert length(unique(row["patch_id"] for row in patch_rows)) == 49
@assert all(parse(Int, row["row_end"]) - parse(Int, row["row_start"]) == 3 for row in patch_rows)
@assert all(parse(Int, row["column_end"]) - parse(Int, row["column_start"]) == 3 for row in patch_rows)
show_table(["Lưới", "Patch/ảnh", "Tổng patch trước lọc"], [["7×7 patch 4×4", "49", "3.430.000"]])


|          Lưới | Patch/ảnh | Tổng patch trước lọc |
| -------------:| ---------:| --------------------:|
| 7×7 patch 4×4 |        49 |            3.430.000 |


## 4. Đánh đổi khi lọc nền

Ngưỡng Q10+ được cố định ở 29; quy tắc giữ patch trong pipeline là có ít nhất một pixel **lớn hơn 29**. So sánh với không lọc và Otsu ở cùng mức `alpha=0` cho thấy Q10+ giảm khối lượng descriptor, trong khi loại ít gradient hơn Otsu. Tổn thất gradient chỉ là phép đo ảnh, không đồng nghĩa với mất thông tin ngữ nghĩa.


In [4]:
filter_rows = read_eda1("06_C4_filter_configurations.csv")
selected_names = ("No-filter", "Q10+ alpha=0.0", "Global Otsu alpha=0.0")
selected_rows = [only(filter(row -> row["configuration"] == name, filter_rows)) for name in selected_names]
filter_summary = [
    [row["configuration"], row["threshold"],
     round(100 * (1 - parse(Float64, row["retained_fraction"])); digits=2),
     round(100 * parse(Float64, row["gradient_loss_ratio"]); digits=2),
     row["empty_images"]]
    for row in selected_rows
]
show_table(["Cấu hình", "Ngưỡng", "Patch bị loại (%)", "Tổn thất gradient (%)", "Ảnh rỗng"], filter_summary)


|              Cấu hình | Ngưỡng | Patch bị loại (%) | Tổn thất gradient (%) | Ảnh rỗng |
| ---------------------:| ------:| -----------------:| ---------------------:| --------:|
|             No-filter |        |               0.0 |                   0.0 |        0 |
|        Q10+ alpha=0.0 |     29 |              37.5 |                  4.41 |        0 |
| Global Otsu alpha=0.0 |     97 |              43.3 |                   8.9 |        0 |


## 5. Descriptor và điểm bàn giao

Raw grayscale patch là baseline gọn, có kích thước cố định 16 chiều và dùng trực tiếp với khoảng cách Euclid của K-means. Sobel/HOG vẫn là phương án đối chứng; SIFT/ORB cần kiểm tra tính đầy đủ và độ đo phù hợp trước khi thay thế baseline. EDA-1 không chứng minh descriptor nào tạo mẫu tốt hơn sau khi phân cụm.

Cấu hình được dùng ở pipeline sau là **patch 4×4 → lọc `max(pixel)>29` → 16 giá trị pixel xám `Float32/255` → K-means → BoVW nhị phân**. K, seed và minsup được khảo sát ở các bước sau; không được suy ra từ EDA-1.


In [5]:
descriptor_rows = read_eda1("09_3_6_descriptor_shortlist.csv")
show_table(["Ứng viên", "Bằng chứng có sẵn", "Điều cần kiểm tra tiếp"], [
    [row["candidate"], row["evidence"], row["validation_needed"]]
    for row in descriptor_rows
])


|              Ứng viên |                                                   Bằng chứng có sẵn |                                                                              Điều cần kiểm tra tiếp |
| ---------------------:| -------------------------------------------------------------------:| ---------------------------------------------------------------------------------------------------:|
|   Raw grayscale patch |                          16 UInt8 dimensions; dense and fixed-size. |                                                            Test normalization and codebook quality. |
| Sobel magnitude patch |           Original-gray edge/texture evidence in outputs 05 and 06. |                                                  Test scale/binning and later transaction patterns. |
|                  SIFT | OpenCV keypoint descriptors; pilot counts and coverage in 09*3*2–4. |                                       Assess zero-feature rate, dimension and full extraction cost. |
|                   ORB |    Binary OpenCV descriptor; pilot counts and coverage in 09*3*2–4. | Use Hamming-compatible quantization or justify a different route; do not presume Euclidean K-means. |
